<a href="https://colab.research.google.com/github/ashikjoel/-Context-Aware-Neural-Recommendation-Engine/blob/ashik_joel/model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [3]:
!pip install -q tf-keras==2.20.1
!pip install -q tensorflow==2.20.0
!pip install -q tensorflow-recommenders==0.7.7

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.8 MB/s eta 0:00:00


In [4]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tensorflow_recommenders as tfrs

print("TensorFlow:", tf.__version__)
print("TFRS:", tfrs.__version__)

TensorFlow: 2.20.0
TFRS: v0.7.7


In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HM Recommendation Evaluation")
    .getOrCreate()
)

In [6]:
BASE_PATH = "/content/drive/MyDrive/Recommendation_Engine"

DATA_PATH = BASE_PATH + "/data/processed"
VOCAB_PATH = BASE_PATH + "/vocabularies"
INDEX_PATH = BASE_PATH + "/final_retrieval_index"

In [7]:
customers = spark.read.parquet(
    f"{DATA_PATH}/customers_clean.parquet"
)

articles = spark.read.parquet(
    f"{DATA_PATH}/articles_clean.parquet"
)

transactions = spark.read.parquet(
    f"{DATA_PATH}/transactions_clean.parquet"
)

In [8]:
print("Customers    :", customers.count())
print("Articles     :", articles.count())
print("Transactions :", transactions.count())

Customers    : 1371980
Articles     : 105542
Transactions : 31788324


In [9]:
retrieval_model = tf.saved_model.load(INDEX_PATH)

print("Retrieval index loaded successfully!")

Retrieval index loaded successfully!


In [10]:
from pyspark.sql.functions import rand

# Sample 1000 customers for evaluation
eval_customers = (
    transactions
    .select("customer_id")
    .distinct()
    .orderBy(rand())
    .limit(1000)
)

print("Evaluation Customers:", eval_customers.count())

Evaluation Customers: 1000


In [11]:
from pyspark.sql import functions as F

ground_truth = (
    transactions
    .join(eval_customers, on="customer_id")
    .groupBy("customer_id")
    .agg(
        F.collect_set("article_id").alias("actual_items")
    )
)

ground_truth.show(5, truncate=False)

+----------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|customer_id                                                     |actual_items                                                                                                                                                                                                                                                                                                                   |
+----------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
ground_truth_pd = ground_truth.toPandas()

print("Evaluation Users:", len(ground_truth_pd))
ground_truth_pd.head()

Evaluation Users: 1000


,customer_id,actual_items
0,00143ec0632e65ee935f083c359df074db8f47cc488934...,"[675408001, 693242003, 803468002, 687535001, 5..."
1,014e43957eb41c42ba0eb4bf75688aa5c071a3194fdaa3...,"[738713029, 521062011, 738713037]"
2,0168f3c92df942e4004dfaf18dd3d389b4a74f16f16964...,"[650759001, 757156001, 827500003, 552346014, 6..."
3,01c02d86295bd76b1e6c225053c2fe0f38d0165f93de89...,"[712711001, 788463002, 720125002, 757303013, 7..."
4,01d93ff34367fe3dcd0e7366e9db3bc103f8a46964eaf4...,"[817358001, 850249003, 752554002, 836862001, 7..."


In [13]:
K = 10

predictions = {}

for customer in ground_truth_pd["customer_id"]:
    _, recs = retrieval_model(tf.constant([customer]))

    predictions[customer] = [
        x.decode("utf-8")
        for x in recs.numpy()[0][:K]
    ]

print("Predictions generated:", len(predictions))

Predictions generated: 1000


In [15]:
import numpy as np
def recall_at_k(actual, predicted, k=10):
    actual = set(actual)
    predicted = set(predicted[:k])

    if len(actual) == 0:
        return 0

    return len(actual & predicted) / len(actual)


recall_scores = []

for _, row in ground_truth_pd.iterrows():

    customer = row["customer_id"]

    actual = row["actual_items"]

    predicted = predictions[customer]

    recall_scores.append(
        recall_at_k(actual, predicted, K)
    )

print("Recall@10:", np.mean(recall_scores))

Recall@10: 0.0


In [16]:
import math

def dcg(actual, predicted, k=10):
    score = 0

    for i, item in enumerate(predicted[:k]):
        if item in actual:
            score += 1 / math.log2(i + 2)

    return score


def ndcg(actual, predicted, k=10):
    ideal = dcg(actual, actual[:k], k)

    if ideal == 0:
        return 0

    return dcg(actual, predicted, k) / ideal


ndcg_scores = []

for _, row in ground_truth_pd.iterrows():

    customer = row["customer_id"]

    actual = row["actual_items"]

    predicted = predictions[customer]

    ndcg_scores.append(
        ndcg(actual, predicted, K)
    )

print("NDCG@10:", np.mean(ndcg_scores))

NDCG@10: 0.0


In [17]:
customer = ground_truth_pd.iloc[0]["customer_id"]

actual = set(ground_truth_pd.iloc[0]["actual_items"])

_, recs = retrieval_model(tf.constant([customer]))

predicted = [x.decode("utf-8") for x in recs.numpy()[0][:10]]

print("Actual:")
print(actual)

print("\nPredicted:")
print(predicted)

print("\nIntersection:")
print(actual.intersection(predicted))

Actual:
{675408001, 507883009, 752016001, 693242003, 691039001, 687535001, 759871001, 417951005, 662980001, 696788001, 158340001, 679977003, 656734005, 742947001, 464297021, 663016001, 751592001, 660712007, 662925002, 739501003, 733362002, 569981011, 560183002, 875724001, 803468002, 751793003, 764358003}

Predicted:
['887181002', '887181002', '887181002', '779612006', '881691001', '881691001', '881691001', '881691001', '881691001', '881691001']

Intersection:
set()


In [18]:
for i in range(5):
    customer = ground_truth_pd.iloc[i]["customer_id"]

    _, recs = retrieval_model(tf.constant([customer]))

    predicted = [x.decode("utf-8") for x in recs.numpy()[0][:10]]

    print(f"\nCustomer {i+1}")
    print(predicted)


Customer 1
['887181002', '887181002', '887181002', '779612006', '881691001', '881691001', '881691001', '881691001', '881691001', '881691001']

Customer 2
['433444007', '433444007', '433444007', '433444007', '433444007', '433444007', '433444007', '433444007', '433444007', '433444007']

Customer 3
['524770002', '524770002', '524770002', '524770002', '524770002', '524770002', '524770002', '903994001', '903994001', '903994001']

Customer 4
['787481005', '787481005', '787481005', '787481005', '787481005', '787481005', '787481005', '787481005', '787481005', '787481005']

Customer 5
['715780002', '715780002', '715780002', '715780002', '715780002', '597818006', '597818006', '597818006', '597818006', '597818006']
